# Merge KannadaGPT-0.6B LoRA into Base Model

This notebook merges the LoRA adapter weights into the base Qwen3-0.6B model, creating a **standalone** KannadaGPT-0.6B that can be loaded directly without PEFT.

**Before**: Need base model + LoRA adapter (~1.2GB + 38MB)

**After**: Single merged model (~1.2GB, no dependencies)

## 1. Install Dependencies

In [ ]:
!pip install -q transformers>=4.40.0 peft accelerate torch huggingface_hub

import transformers
print(f"transformers: {transformers.__version__}")

## 2. Configuration

In [ ]:
import torch
import os

# Model configuration
BASE_MODEL = "Qwen/Qwen3-0.6B"
LORA_ADAPTER = "Mithun501/KannadaGPT-0.6B"  # Your LoRA adapter on HuggingFace
OUTPUT_DIR = "./KannadaGPT-0.6B-merged"

# For uploading merged model (optional)
UPLOAD_TO_HF = True  # Set to False if you don't want to upload
HF_REPO_MERGED = "Mithun501/KannadaGPT-0.6B-merged"  # New repo for merged model

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Base Model: {BASE_MODEL}")
print(f"LoRA Adapter: {LORA_ADAPTER}")
print(f"Output Directory: {OUTPUT_DIR}")
print(f"\nGPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

## 3. HuggingFace Login (for uploading)

In [ ]:
from huggingface_hub import login, HfApi

# Option 1: Use Colab secrets (recommended)
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    print("Using HF token from Colab secrets")
except:
    # Option 2: Enter token manually
    HF_TOKEN = ""  # Paste your token here if not using secrets
    print("Enter your HF token above or use Colab secrets")

if HF_TOKEN:
    login(token=HF_TOKEN)
    print("Logged in to HuggingFace!")
else:
    print("No HF token - will save locally only")
    UPLOAD_TO_HF = False

## 4. Load Base Model and LoRA Adapter

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# Load base model
print(f"Loading base model: {BASE_MODEL}...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)
print(f"Base model loaded!")

# Load tokenizer
print(f"\nLoading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(LORA_ADAPTER, trust_remote_code=True)
print("Tokenizer loaded!")

# Load LoRA adapter
print(f"\nLoading LoRA adapter: {LORA_ADAPTER}...")
model = PeftModel.from_pretrained(base_model, LORA_ADAPTER)
print("LoRA adapter loaded!")

## 5. Merge Weights

In [ ]:
print("Merging LoRA weights into base model...")
print("This combines the adapter with the base model.")

merged_model = model.merge_and_unload()

print("\nMerge complete!")
print(f"Model type: {type(merged_model).__name__}")
print(f"Parameters: {merged_model.num_parameters():,}")

## 6. Save Merged Model Locally

In [ ]:
print(f"Saving merged model to {OUTPUT_DIR}...")

# Save model
merged_model.save_pretrained(OUTPUT_DIR, safe_serialization=True)

# Save tokenizer
tokenizer.save_pretrained(OUTPUT_DIR)

print("\nSaved files:")
for f in os.listdir(OUTPUT_DIR):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / (1024*1024)
    print(f"  {f}: {size:.1f} MB")

## 7. Test Merged Model

In [ ]:
def test_model(prompt, model, tokenizer):
    """Test generation with the merged model"""
    messages = [{"role": "user", "content": prompt}]
    
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )
    
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.7,
            top_p=0.8,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Test prompts
test_prompts = [
    "ಭಾರತದ ರಾಜಧಾನಿ ಯಾವುದು?",
    "ಬೆಂಗಳೂರಿನ ಬಗ್ಗೆ ಹೇಳಿ",
]

print("Testing merged model:\n")
for prompt in test_prompts:
    print(f"Q: {prompt}")
    response = test_model(prompt, merged_model, tokenizer)
    print(f"A: {response}\n")
    print("-" * 50)

## 8. Upload to HuggingFace (Optional)

In [ ]:
if UPLOAD_TO_HF and HF_TOKEN:
    # Create README for merged model
    readme = '''---
license: apache-2.0
language:
- kn
- en
base_model: Qwen/Qwen3-0.6B
tags:
- kannada
- qwen3
- merged
- indian-languages
library_name: transformers
pipeline_tag: text-generation
---

# KannadaGPT-0.6B (Merged)

**Standalone** Kannada language model - LoRA weights merged into base model.

No PEFT/adapter required - load directly with transformers!

## Usage

```python
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained(
    "Mithun501/KannadaGPT-0.6B-merged",
    torch_dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained("Mithun501/KannadaGPT-0.6B-merged")

messages = [{"role": "user", "content": "ಭಾರತದ ರಾಜಧಾನಿ ಯಾವುದು?"}]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=256)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))
```

## Model Info

| Property | Value |
|----------|-------|
| Base Model | Qwen/Qwen3-0.6B |
| Language | Kannada |
| Type | Merged (standalone) |
| Parameters | 0.6B |
| Checkpoint | 4500 steps |

## Author

[Mithun501](https://github.com/mithun50)
'''
    
    with open(f"{OUTPUT_DIR}/README.md", "w") as f:
        f.write(readme)
    
    print(f"Uploading to {HF_REPO_MERGED}...")
    
    api = HfApi()
    api.create_repo(repo_id=HF_REPO_MERGED, exist_ok=True)
    
    api.upload_folder(
        folder_path=OUTPUT_DIR,
        repo_id=HF_REPO_MERGED,
        repo_type="model",
        commit_message="Upload merged KannadaGPT-0.6B model"
    )
    
    print(f"\nUploaded to: https://huggingface.co/{HF_REPO_MERGED}")
else:
    print("Skipping upload - model saved locally only")
    print(f"Model location: {OUTPUT_DIR}")

## 9. Download Merged Model (Colab)

In [ ]:
# Zip and download the merged model
import shutil

zip_name = "KannadaGPT-0.6B-merged"
shutil.make_archive(zip_name, 'zip', OUTPUT_DIR)

print(f"Created: {zip_name}.zip")
print("Download from Files panel on the left →")

# Auto-download in Colab
try:
    from google.colab import files
    files.download(f"{zip_name}.zip")
except:
    pass

---

## Done!

Your merged model is ready:

**Before (LoRA):**
```python
from peft import PeftModel
base = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-0.6B")
model = PeftModel.from_pretrained(base, "Mithun501/KannadaGPT-0.6B")
```

**After (Merged):**
```python
model = AutoModelForCausalLM.from_pretrained("Mithun501/KannadaGPT-0.6B-merged")
```

Much simpler!